# Gated comparison on Colab

Select a GPU runtime for neural stages. The tabular candidate remains CPU-only. Each invocation reruns the mandatory preflight and writes heartbeats and artifacts directly to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPOSITORY = 'https://github.com/RosarioDiBartolo/Sports-Predictions-Lab.git'
COMMIT_SHA = 'REPLACE_WITH_TESTED_COMMIT_SHA'
BUNDLE_DIR = '/content/drive/MyDrive/Sports Predictions Lab/datasets/REPLACE_WITH_DATASET_VERSION'
RUN_ROOT = '/content/drive/MyDrive/Sports Predictions Lab/runs'
CANDIDATE = 'dixon_coles_shared_encoder_pooling_gated'
ABLATION = 'base'
EPOCHS = 10
MAX_ITER = 20
DEVICE = 'cuda'

In [ ]:
import pathlib
import shutil
import subprocess

checkout = pathlib.Path('/content/sports-predictions-lab')
if checkout.exists():
    shutil.rmtree(checkout)
subprocess.run(['git', 'clone', REPOSITORY, str(checkout)], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', COMMIT_SHA], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-e', str(checkout)], check=True)

In [ ]:
import torch
from football_odds.modeling.training_bundle import verify_training_bundle

manifest = verify_training_bundle(pathlib.Path(BUNDLE_DIR))
print('Dataset:', manifest['dataset_version'])
print('Commit:', COMMIT_SHA)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before neural training.')

In [ ]:
command = [
    'odds-lab', '--project-dir', BUNDLE_DIR, 'model', 'compare',
    '--candidate', CANDIDATE, '--ablation', ABLATION,
    '--epochs', str(EPOCHS), '--max-iter', str(MAX_ITER),
    '--device', DEVICE, '--run-root', RUN_ROOT,
]
print(' '.join(command))
subprocess.run(command, check=True)